In [1]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/seoultechpse/fenicsx-colab.git"
ROOT = Path("/content")
REPO_DIR = ROOT / "fenicsx-colab"

subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

USE_COMPLEX = False  # <--- Set True ONLY if you need complex PETSc
USE_CLEAN = False    # <--- Set True to remove existing environment

opts_str = " ".join(
  [o for c, o in [(USE_COMPLEX, "--complex"), (USE_CLEAN, "--clean")] if c]
)

get_ipython().run_line_magic("run", f"{REPO_DIR / 'setup_fenicsx.py'} {opts_str}")

🔧 FEniCSx Setup Configuration
PETSc type      : real
Clean install   : False

⚠️  Google Drive not mounted — using local cache (/content)

🔧 Installing FEniCSx environment...

🔍 Verifying PETSc type...
✅ Installed: Real PETSc (float64)

✨ Loading FEniCSx Jupyter magic... %%fenicsx registered

✅ FEniCSx setup complete!

Next steps:
  1. Run %%fenicsx --info to verify installation
  2. Use %%fenicsx in cells to run FEniCSx code
  3. Use -np N for parallel execution (e.g., %%fenicsx -np 4)

📌 Note: Real PETSc is installed
   - Recommended for most FEM problems
   - For complex problems, reinstall with --complex


---

In [2]:
%%fenicsx

"""
FEniCSx PETSc Solving Interface - Linear Elasticity
Extracted from Colab notebook

This script demonstrates solving linear elasticity problems using FEniCSx with PETSc solvers.
It includes:
- Beam deformation analysis
- PETSc direct solver usage
- Lifting technique for boundary conditions
- VTK output for ParaView visualization
"""

# Import required libraries
from mpi4py import MPI
from petsc4py import PETSc

import numpy as np

import dolfinx
import dolfinx.fem.petsc
import ufl
from dolfinx.io import VTKFile

print(f"DOLFINx version: {dolfinx.__version__}")
print(f"Communicator size: {MPI.COMM_WORLD.size}")


# ============================================================================
# 1. MESH CREATION
# ============================================================================
# Create mesh
L = 10.0  # Length
W = 3.0   # Width
H = 3.0   # Height

mesh = dolfinx.mesh.create_box(
    MPI.COMM_WORLD,
    [[0.0, 0.0, 0.0], [L, W, H]],
    [15, 7, 7],
    cell_type=dolfinx.mesh.CellType.hexahedron,
)

print(f"Mesh information:")
print(f"  - Number of cells: {mesh.topology.index_map(mesh.topology.dim).size_local}")
print(f"  - Spatial dimension: {mesh.geometry.dim}")


# ============================================================================
# 2. FUNCTION SPACE DEFINITION
# ============================================================================
# Define function space (vector Lagrange 2nd order elements)
tdim = mesh.topology.dim
V = dolfinx.fem.functionspace(mesh, ("Lagrange", 2, (mesh.geometry.dim,)))

print(f"Function space information:")
print(f"  - Number of DOFs: {V.dofmap.index_map.size_global * V.dofmap.index_map_bs}")


# ============================================================================
# 3. BOUNDARY FACET IDENTIFICATION
# ============================================================================
# Create mesh topology
mesh.topology.create_connectivity(tdim - 1, tdim)

# Find all exterior facets
boundary_facets = dolfinx.mesh.exterior_facet_indices(mesh.topology)
print(f"Number of exterior facets: {len(boundary_facets)}")


# Classify facets by boundary condition

# 1. Left end (x = 0): Clamped boundary
def left_facets(x):
    return np.isclose(x[0], 0.0)

clamped_facets = dolfinx.mesh.locate_entities_boundary(mesh, tdim - 1, left_facets)
print(f"Number of clamped facets: {len(clamped_facets)}")

# 2. Right end (x = L): Prescribed displacement (using lambda function)
prescribed_facets = dolfinx.mesh.locate_entities_boundary(
    mesh, tdim - 1, lambda x: np.isclose(x[0], L)
)
print(f"Number of prescribed displacement facets: {len(prescribed_facets)}")

# 3. Remaining: Free surfaces
free_facets = np.setdiff1d(
    boundary_facets,
    np.union1d(clamped_facets, prescribed_facets)
)
print(f"Number of free surfaces: {len(free_facets)}")


# Create mesh markers
num_facets = mesh.topology.index_map(tdim - 1).size_local
markers = np.zeros(num_facets, dtype=np.int32)

# Assign integer markers to each boundary type
clamped = 1
prescribed = 2
free = 3

markers[clamped_facets] = clamped
markers[prescribed_facets] = prescribed
markers[free_facets] = free

facet_marker = dolfinx.mesh.meshtags(
    mesh, tdim - 1,
    np.arange(num_facets, dtype=np.int32),
    markers
)

print("Mesh markers created successfully")


# ============================================================================
# 4. MATERIAL CONSTANTS AND VARIATIONAL FORMULATION
# ============================================================================
# Define material constants
E = dolfinx.fem.Constant(mesh, 1.4e3)  # Young's modulus
nu = dolfinx.fem.Constant(mesh, 0.3)   # Poisson's ratio

# Compute Lamé parameters
mu = E / (2.0 * (1.0 + nu))
lmbda = E * nu / ((1.0 + nu) * (1.0 - 2.0 * nu))

# Define loading
f = dolfinx.fem.Constant(mesh, (0.0, 0.0, 0.0))  # Body force
T_0 = dolfinx.fem.Constant(mesh, (0.0, 0.0, 0.0))  # Traction

print("Material constants and loading defined")


# Define strain tensor
def epsilon(u):
    """Symmetric strain tensor"""
    return ufl.sym(ufl.grad(u))

# Define stress tensor
def sigma(u):
    """Stress tensor (isotropic elasticity)"""
    return 2.0 * mu * epsilon(u) + lmbda * ufl.tr(epsilon(u)) * ufl.Identity(len(u))

print("Stress-strain relationship defined")


# Define weak form
ds = ufl.Measure("ds", domain=mesh, subdomain_data=facet_marker)
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

# Bilinear form
a = ufl.inner(sigma(u), epsilon(v)) * ufl.dx

# Linear form
L = ufl.inner(f, v) * ufl.dx + ufl.inner(T_0, v) * ds(3)

print("Variational formulation defined")


# ============================================================================
# 5. DIRICHLET BOUNDARY CONDITIONS AND LIFTING
# ============================================================================
# Locate degrees of freedom on boundaries
clamped_dofs = dolfinx.fem.locate_dofs_topological(
    V, facet_marker.dim, facet_marker.find(clamped)
)

displaced_dofs = dolfinx.fem.locate_dofs_topological(
    V, facet_marker.dim, facet_marker.find(prescribed)
)

print(f"Number of clamped DOFs: {len(clamped_dofs)}")
print(f"Number of prescribed displacement DOFs: {len(displaced_dofs)}")


# Set boundary values
u_prescribed = dolfinx.fem.Constant(mesh, (0.0, 0.0, -H / 2))
u_clamped = dolfinx.fem.Constant(mesh, (0.0, 0.0, 0.0))

# Create Dirichlet boundary condition objects
bcs = [
    dolfinx.fem.dirichletbc(u_clamped, clamped_dofs, V),
    dolfinx.fem.dirichletbc(u_prescribed, displaced_dofs, V),
]

print("Dirichlet boundary conditions created")


# Lifting procedure: Method using UFL action
g = dolfinx.fem.Function(V)
g.x.array[:] = 0
dolfinx.fem.set_bc(g.x.array, bcs)
g.x.scatter_forward()

# Modify RHS: L → L - a(g, v)
L_lifted = L - ufl.action(a, g)

print("Lifting completed: Non-homogeneous BC converted to homogeneous BC")


# ============================================================================
# 6. SOLVE WITH PETSC (Method 1: Direct API)
# ============================================================================
# Assemble matrix and vector
a_compiled = dolfinx.fem.form(a)
A = dolfinx.fem.petsc.assemble_matrix(a_compiled, bcs=bcs)
A.assemble()

b = dolfinx.fem.petsc.assemble_vector(dolfinx.fem.form(L_lifted))
dolfinx.fem.petsc.set_bc(b, bcs)
b.ghostUpdate(addv=PETSc.InsertMode.INSERT_VALUES, mode=PETSc.ScatterMode.FORWARD)

print("PETSc matrix/vector assembly completed")
print(f"  - Matrix size: {A.getSize()}")
print(f"  - Vector size: {b.getSize()}")


# Create and configure PETSc KSP solver
ksp = PETSc.KSP().create(mesh.comm)
ksp.setType("preonly")  # Preconditioner only
ksp.getPC().setType("lu")  # LU factorization
ksp.getPC().setFactorSolverType("mumps")  # MUMPS direct solver

# Set operator
ksp.setOperators(A)

print("PETSc solver configured")

# Solve linear system
uh = dolfinx.fem.Function(V)
ksp.solve(b, uh.x.petsc_vec)

# Check convergence
converged = ksp.getConvergedReason()
iterations = ksp.getIterationNumber()

print(f"\nSolution computed:")
print(f"  - Convergence reason: {converged} (positive means success)")
print(f"  - Number of iterations: {iterations}")

assert converged > 0, "Solver did not converge!"

uh.x.scatter_forward()

# Free memory
ksp.destroy()
b.destroy()


# ============================================================================
# 7. SOLVE WITH PETSC (Method 2: LinearProblem Wrapper)
# ============================================================================
# Convenient interface
u_new = dolfinx.fem.Function(V)

options = {
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
    "ksp_error_if_not_converged": True,
}

problem = dolfinx.fem.petsc.LinearProblem(
    a, L, bcs=bcs, u=u_new,
    petsc_options=options,
    petsc_options_prefix="elasticity_"
)

problem.solve()

print("\nSolution computed with LinearProblem")
print(f"Convergence reason: {problem.solver.getConvergedReason()}")

# Compare results from both methods
np.testing.assert_allclose(uh.x.array, u_new.x.array, atol=1e-12)
print("Results from both methods match!")


# ============================================================================
# 8. RESULT ANALYSIS
# ============================================================================
# Displacement statistics
displacement = uh.x.array.reshape(-1, 3)

print("\nDisplacement statistics:")
print(f"  Maximum x-displacement: {np.max(np.abs(displacement[:, 0])):.6f}")
print(f"  Maximum y-displacement: {np.max(np.abs(displacement[:, 1])):.6f}")
print(f"  Maximum z-displacement: {np.max(np.abs(displacement[:, 2])):.6f}")
print(f"  Maximum displacement magnitude: {np.max(np.linalg.norm(displacement, axis=1)):.6f}")


# ============================================================================
# 9. SAVE RESULTS FOR PARAVIEW
# ============================================================================
# Save displacement field
with VTKFile(mesh.comm, "displacement.pvd", "w") as vtk:
    vtk.write_function(uh, 0.0)

print("Displacement field saved to 'displacement.pvd'")


# Compute and save stress tensor
# Project to DG (Discontinuous Galerkin) space
W = dolfinx.fem.functionspace(mesh, ("DG", 1, (3, 3)))  # Tensor space

stress_expr = dolfinx.fem.Expression(
    sigma(uh),
    W.element.interpolation_points
)

stress_field = dolfinx.fem.Function(W)
stress_field.interpolate(stress_expr)

# Save stress field
with VTKFile(mesh.comm, "stress.pvd", "w") as vtk:
    vtk.write_function(stress_field, 0.0)

print("Stress field saved to 'stress.pvd'")


# Compute Von Mises stress (scalar)
# Von Mises stress is used to evaluate material yield condition
def von_mises_stress(s):
    """Compute Von Mises stress"""
    return ufl.sqrt(
        3./2. * ufl.inner(s - (1./3.) * ufl.tr(s) * ufl.Identity(3),
                          s - (1./3.) * ufl.tr(s) * ufl.Identity(3))
    )

# Scalar DG space
W_scalar = dolfinx.fem.functionspace(mesh, ("DG", 1))

vm_expr = dolfinx.fem.Expression(
    von_mises_stress(sigma(uh)),
    W_scalar.element.interpolation_points
)

vm_field = dolfinx.fem.Function(W_scalar)
vm_field.interpolate(vm_expr)

# Save Von Mises stress
with VTKFile(mesh.comm, "von_mises_stress.pvd", "w") as vtk:
    vtk.write_function(vm_field, 0.0)

print("Von Mises stress saved to 'von_mises_stress.pvd'")
print(f"Maximum Von Mises stress: {np.max(vm_field.x.array):.2f}")


# ============================================================================
# 10. (OPTIONAL) NONLINEAR SOLVER EXAMPLE
# ============================================================================
# Define nonlinear form
uh_nonlinear = dolfinx.fem.Function(V)

# Replace trial function with unknown function
F = a - L
F = ufl.replace(F, {u: uh_nonlinear})

# Create NonlinearProblem
nonlinear_options = {
    "snes_type": "newtonls",
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
}

nonlinear_problem = dolfinx.fem.petsc.NonlinearProblem(
    F, uh_nonlinear, bcs=bcs,
    petsc_options=nonlinear_options,
    petsc_options_prefix="elasticity_nonlinear_"
)

nonlinear_problem.solve()

converged = nonlinear_problem.solver.getConvergedReason()
num_iterations = nonlinear_problem.solver.getIterationNumber()

print(f"\nNonlinear solver results:")
print(f"  Convergence reason: {converged}")
print(f"  Number of iterations: {num_iterations} (1 iteration for linear problem)")

# Compare results
np.testing.assert_allclose(uh.x.array, uh_nonlinear.x.array, atol=1e-10)
print("Linear/nonlinear solver results match!")


# ============================================================================
# SCRIPT COMPLETED
# ============================================================================
print("\n" + "="*70)
print("Analysis completed successfully!")
print("="*70)
print("\nGenerated files:")
print("  - displacement.pvd (and .vtu)")
print("  - stress.pvd (and .vtu)")
print("  - von_mises_stress.pvd (and .vtu)")
print("\nOpen these files in ParaView for visualization.")
print("="*70)

DOLFINx version: 0.10.0
Communicator size: 1
Mesh information:
  - Number of cells: 735
  - Spatial dimension: 3
Function space information:
  - Number of DOFs: 20925
Number of exterior facets: 518
Number of clamped facets: 49
Number of prescribed displacement facets: 49
Number of free surfaces: 420
Mesh markers created successfully
Material constants and loading defined
Stress-strain relationship defined
Variational formulation defined
Number of clamped DOFs: 225
Number of prescribed displacement DOFs: 225
Dirichlet boundary conditions created
Lifting completed: Non-homogeneous BC converted to homogeneous BC
PETSc matrix/vector assembly completed
  - Matrix size: (20925, 20925)
  - Vector size: 20925
PETSc solver configured

Solution computed:
  - Convergence reason: 4 (positive means success)
  - Number of iterations: 1

Solution computed with LinearProblem
Convergence reason: 4
Results from both methods match!

Displacement statistics:
  Maximum x-displacement: 0.273870
  Maximum y-

In [3]:
!mkdir -p paraview
!mv *.pvd *.pvtu *.vtu paraview/
!zip -r paraview.zip paraview

from google.colab import files
files.download("paraview.zip")

  adding: paraview/ (stored 0%)
  adding: paraview/von_mises_stress_p0_000000.vtu (deflated 82%)
  adding: paraview/von_mises_stress000000.pvtu (deflated 61%)
  adding: paraview/von_mises_stress.pvd (deflated 28%)
  adding: paraview/stress000000.pvtu (deflated 61%)
  adding: paraview/displacement000000.pvtu (deflated 61%)
  adding: paraview/displacement_p0_000000.vtu (deflated 73%)
  adding: paraview/stress.pvd (deflated 28%)
  adding: paraview/displacement.pvd (deflated 29%)
  adding: paraview/stress_p0_000000.vtu (deflated 72%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>